# Tpot

In [1]:
!pip install tpot

In [1]:
import sys, os, os.path, pickle, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from category_encoders.cat_boost import CatBoostEncoder

from sklearn import preprocessing
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, make_scorer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline, make_union
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

from tpot import TPOTClassifier
from tpot.builtins import StackingEstimator
from tpot.export_utils import set_param_recursive

warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv('./train.csv')
test = pd.read_csv('./test.csv')

def get_x_y(df):
    if 'class' in df.columns:
        df_x = df.drop(columns=['id', 'class'])
        df_y = df['class']
        return df_x, df_y
    else:
        df_x = df.drop(columns=['id'])
        return df_x
    
train_x, train_y = get_x_y(train)
test_x = get_x_y(test)

class_le = preprocessing.LabelEncoder()
snp_le = preprocessing.LabelEncoder()
snp_col = [f'SNP_{str(x).zfill(2)}' for x in range(1,16)]

snp_data = []
for col in snp_col:
    snp_data += list(train_x[col].values)
    
train_y = class_le.fit_transform(train_y)
snp_le.fit(snp_data)

for col in train_x.columns:
    if col in snp_col:
        train_x[col] = snp_le.transform(train_x[col])
        test_x[col] = snp_le.transform(test_x[col])

train_x['label']=train_y

train_x=train_x.iloc[:,3:]
test_x=test_x.iloc[:,3:]


cat_train_df = train_x.copy()
cat_test_df = train_y.copy()

ce = CatBoostEncoder()

columns_list=list(cat_train_df.columns)

columns_list.remove('label')

train_x[columns_list] = ce.fit_transform(train_x[columns_list], train_x['label'])
test_x[columns_list] = ce.transform(test_x[columns_list])

In [3]:
X = train_x.drop(columns=['label'])
y = train_x['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2023)

In [4]:
X_train.head()

,trait,SNP_01,SNP_02,SNP_03,SNP_04,SNP_05,SNP_06,SNP_07,SNP_08,SNP_09,SNP_10,SNP_11,SNP_12,SNP_13,SNP_14,SNP_15
200,1,1,5,0,5,0,5,5,0,5,1,1,4,5,2,0
180,1,0,5,0,4,2,1,4,0,0,1,5,5,5,3,0
79,1,0,5,0,4,0,5,5,0,0,0,5,5,5,2,5
195,2,5,1,2,4,3,5,0,4,0,5,0,0,1,0,0
21,2,5,1,3,0,3,1,0,5,0,5,5,0,1,0,4


In [5]:
def my_custom_accuracy(y_true, y_pred):
    score = f1_score(y_true, y_pred, average='macro')
    return score

In [6]:
my_custom_scorer = make_scorer(my_custom_accuracy, greater_is_better=True)

In [29]:
tpot = TPOTClassifier(
    generations=100,
    # template='Selector-Transformer-XGBClassifier',
    # config_dict=classifier_config_dict,
    population_size=100,
    mutation_rate=0.85, 
    crossover_rate=0.15, 
    verbosity=2, 
    random_state=42,
    scoring=my_custom_scorer,
    max_eval_time_mins=5
)

tpot.fit(X_train, y_train)

print(tpot.score(X_test, y_test))

Optimization Progress:   0%|          | 0/10100 [00:00<?, ?pipeline/s]


Generation 1 - Current best internal CV score: 0.9547892514167025


TPOT closed during evaluation in one generation.


TPOT closed prematurely. Will use the current best pipeline.

Best pipeline: MLPClassifier(LinearSVC(input_matrix, C=0.001, dual=False, loss=squared_hinge, penalty=l2, tol=0.001), alpha=0.1, learning_rate_init=0.001)
0.9650793650793652


In [24]:
results = tpot.predict(test_x)

In [25]:
submit = pd.read_csv('./sample_submission.csv')

In [26]:
submit['class'] = class_le.inverse_transform(results)

In [27]:
submit.to_csv('./TPOTClassifier.csv', index=False)

In [28]:
tpot.export('./tpot_digits_pipeline_test.py')

In [ ]:
exported_pipeline = make_pipeline(
    VarianceThreshold(threshold=0.01),
    PCA(iterated_power=6, svd_solver="randomized"),
    XGBClassifier(
        learning_rate=1.0, 
        max_depth=10, 
        min_child_weight=1, 
        n_estimators=100, 
        n_jobs=1, 
        subsample=0.6000000000000001, 
        verbosity=0
    )
)

In [ ]:
# Average CV score on the training set was: 0.9671786451394295
exported_pipeline = GradientBoostingClassifier(
    learning_rate=0.1, 
    max_depth=4, 
    max_features=0.1, 
    min_samples_leaf=1, 
    min_samples_split=6, 
    n_estimators=100, 
    subsample=0.8
)
# Fix random state in exported estimator
if hasattr(exported_pipeline, 'random_state'):
    setattr(exported_pipeline, 'random_state', 42)

In [ ]:
X

In [35]:
exported_pipeline.fit(X, train_y)
results = exported_pipeline.predict(test_x)

ValueError: Negative values in data passed to MultinomialNB (input X)

In [ ]:
submit = pd.read_csv('./sample_submission.csv')

In [ ]:
submit['class'] = class_le.inverse_transform(results)

In [ ]:
submit.to_csv('./tpot_test.csv', index=False)

In [ ]:
submit

In [ ]:
# # Average CV score on the training set was: 0.9723435503043347
# exported_pipeline = make_pipeline(
#     StackingEstimator(
#         estimator = GradientBoostingClassifier(
#             learning_rate=0.5, 
#             max_depth=7, 
#             max_features=0.8, 
#             min_samples_leaf=9, 
#             min_samples_split=5, 
#             n_estimators=100, 
#             subsample=0.2)
#     ),
    
#     GradientBoostingClassifier(
#         learning_rate=0.1, 
#         max_depth=9, 
#         max_features=0.8500000000000001, 
#         min_samples_leaf=16, 
#         min_samples_split=20, 
#         n_estimators=100, 
#         subsample=0.6000000000000001
#     )
# )
# # Fix random state for all the steps in exported pipeline
# set_param_recursive(exported_pipeline.steps, 'random_state', 42)

In [55]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA

# Average CV score on the training set was: 0.9631683010073815
exported_pipeline = make_pipeline(
    VarianceThreshold(threshold=0.5),
    # PCA(iterated_power=6, svd_solver="randomized"),
    StackingEstimator(
        estimator = DecisionTreeClassifier(
            criterion="gini", 
            max_depth=2, 
            min_samples_leaf=12, 
            min_samples_split=9)
    ),
    StackingEstimator(
        estimator = GradientBoostingClassifier(
            learning_rate=0.5, 
            max_depth=7, 
            max_features=0.8, 
            min_samples_leaf=9, 
            min_samples_split=5, 
            n_estimators=100, 
            subsample=0.2)
    ),
    StackingEstimator(
        estimator = MultinomialNB(
            alpha=1.0, 
            fit_prior=True)
    ),
    StackingEstimator(
        estimator = LinearSVC(
            C=15.0, 
            dual=True,
            loss="squared_hinge", 
            penalty="l2", 
            tol=0.0001)
    ),
    GradientBoostingClassifier(
        learning_rate=0.1, 
        max_depth=5, 
        max_features=0.25, 
        min_samples_leaf=18, 
        min_samples_split=14, 
        n_estimators=100, 
        subsample=0.5
    )
)
# Fix random state for all the steps in exported pipeline
set_param_recursive(exported_pipeline.steps, 'random_state', 42)

In [56]:
target = train_x['label']
new_train_df = train_x.iloc[:,:-1]

In [57]:
exported_pipeline.fit(new_train_df, target)
results = exported_pipeline.predict(test_x)

In [58]:
submit = pd.read_csv('./sample_submission.csv')

In [59]:
submit['class'] = class_le.inverse_transform(results)

In [60]:
submit['class'] = results

In [61]:
submit['class'] = submit['class'].apply(lambda x: 'A' if x == 0 else ( 'B' if x == 1 else 'C' ))

In [62]:
submit.to_csv('./tpot.csv', index=False)